# Generate `oil_network` HTML explorers

Reads the current state of `oil_network` (assets, variables, timeseries, `scenario_resolved_values`) and writes every self-contained HTML explorer in this directory. Runs **after** [initialize_oil_network.ipynb](initialize_oil_network.ipynb) — that orchestrator builds the database and resolves the scenario but does not produce HTMLs.

## Outputs

| Generator | Output | What it shows |
|---|---|---|
| [make_map.py](make_map.py) | [oil_network_map.html](oil_network_map.html) | Geographic map of 195 physical assets + 376 flow edges on natural-earth projection |
| [make_hierarchy_explorer.py](make_hierarchy_explorer.py) | [oil_network_hierarchy.html](oil_network_hierarchy.html) | Drill-down tree from roots to physical leaves |
| [make_balance_ui.py](make_balance_ui.py) | [oil_network_balance.html](oil_network_balance.html) | Per-node balance equation (P/C/I/O/B/S/ΔS) at a selected date |
| [make_map_resolver_ui.py](make_map_resolver_ui.py) | [oil_network_map_resolver.html](oil_network_map_resolver.html) | Map with tooltip counts from `scenario_resolved_values` |
| [make_hierarchy_resolver_ui.py](make_hierarchy_resolver_ui.py) | [oil_network_hierarchy_resolver.html](oil_network_hierarchy_resolver.html) | Hierarchy fed by the resolver instead of inline SQL |
| [make_balance_resolver_ui.py](make_balance_resolver_ui.py) | [oil_network_balance_resolver.html](oil_network_balance_resolver.html) | Balance view fed by `scenario_resolved_values` |
| [make_explorer.py](make_explorer.py) | [oil_network_explorer.html](oil_network_explorer.html) | Variable-type selector + tree + Plotly time-series chart |

Each generator is independent and idempotent. Toggle individual stages on/off via the flags in the next cell.

## Stage flags

Set each to `True` (run) or `False` (skip).

In [1]:
RUN_MAP                = True   # oil_network_map.html
RUN_HIERARCHY          = True   # oil_network_hierarchy.html
RUN_BALANCE            = True   # oil_network_balance.html
RUN_MAP_RESOLVER       = True   # oil_network_map_resolver.html
RUN_HIERARCHY_RESOLVER = True   # oil_network_hierarchy_resolver.html
RUN_BALANCE_RESOLVER   = True   # oil_network_balance_resolver.html
RUN_EXPLORER           = True   # oil_network_explorer.html

## Run

In [2]:
import subprocess
import sys
from datetime import datetime
from pathlib import Path

GENERATORS = [
    ("map",                RUN_MAP,                "make_map.py",                  "oil_network_map.html"),
    ("hierarchy",          RUN_HIERARCHY,          "make_hierarchy_explorer.py",   "oil_network_hierarchy.html"),
    ("balance",            RUN_BALANCE,            "make_balance_ui.py",           "oil_network_balance.html"),
    ("map (resolver)",     RUN_MAP_RESOLVER,       "make_map_resolver_ui.py",      "oil_network_map_resolver.html"),
    ("hierarchy (resolver)", RUN_HIERARCHY_RESOLVER, "make_hierarchy_resolver_ui.py", "oil_network_hierarchy_resolver.html"),
    ("balance (resolver)", RUN_BALANCE_RESOLVER,   "make_balance_resolver_ui.py",  "oil_network_balance_resolver.html"),
    ("explorer",           RUN_EXPLORER,           "make_explorer.py",             "oil_network_explorer.html"),
]

TIMEOUT_SEC = 600


def stamp():
    return f"[{datetime.now():%H:%M:%S}]"


def run_script(script_path):
    return subprocess.run(
        [sys.executable, script_path],
        capture_output=True, text=True, timeout=TIMEOUT_SEC,
    )


results = []
started = datetime.now()

for label, flag, script, html in GENERATORS:
    if not flag:
        print(f"{stamp()} ⏭  {label:<22} SKIPPED (flag = False)")
        results.append((label, "skipped", html, None))
        continue
    if not Path(script).exists():
        print(f"{stamp()} ✗  {label:<22} FILE MISSING ({script})")
        results.append((label, "missing", html, script))
        continue
    t0 = datetime.now()
    print(f"{stamp()} ▶  {label:<22} running {script}...")
    res = run_script(script)
    dt = (datetime.now() - t0).total_seconds()
    if res.returncode != 0:
        tail = res.stderr.strip().splitlines()[-15:]
        print(f"{stamp()} ✗  {label:<22} FAILED (exit {res.returncode}, {dt:.1f}s)")
        results.append((label, "failed", html, "\n".join(tail)))
    else:
        out_path = Path(html)
        size_kb = out_path.stat().st_size // 1024 if out_path.exists() else 0
        print(f"{stamp()} ✓  {label:<22} done  {html}  ({size_kb} KB, {dt:.1f}s)")
        results.append((label, "ok", html, f"{size_kb} KB, {dt:.1f}s"))

elapsed = (datetime.now() - started).total_seconds()
print()
print("=" * 80)
print(f"HTML PIPELINE SUMMARY  (total elapsed {elapsed:.1f}s)")
print("=" * 80)
for label, status, html, info in results:
    icon = {"ok": "✓", "skipped": "⏭", "missing": "✗", "failed": "✗"}[status]
    suffix = f"  ({info})" if info else ""
    print(f"  {icon} {label:<22} {status:<8} {html}{suffix}")

failed = [r for r in results if r[1] == "failed"]
if failed:
    print()
    print("=" * 80)
    print("FAILURES — last 15 lines of each generator's stderr")
    print("=" * 80)
    for label, _, _, err in failed:
        print()
        print(f"--- {label} ---")
        print(err)

[21:35:29] ▶  map                    running make_map.py...


[21:35:29] ✓  map                    done  oil_network_map.html  (137 KB, 0.4s)
[21:35:29] ▶  hierarchy              running make_hierarchy_explorer.py...


[21:35:29] ✓  hierarchy              done  oil_network_hierarchy.html  (771 KB, 0.3s)
[21:35:29] ▶  balance                running make_balance_ui.py...


[21:35:30] ✓  balance                done  oil_network_balance.html  (453 KB, 0.5s)
[21:35:30] ▶  map (resolver)         running make_map_resolver_ui.py...


[21:35:30] ✓  map (resolver)         done  oil_network_map_resolver.html  (136 KB, 0.3s)
[21:35:30] ▶  hierarchy (resolver)   running make_hierarchy_resolver_ui.py...


[21:35:30] ✓  hierarchy (resolver)   done  oil_network_hierarchy_resolver.html  (602 KB, 0.3s)
[21:35:30] ▶  balance (resolver)     running make_balance_resolver_ui.py...


[21:35:31] ✓  balance (resolver)     done  oil_network_balance_resolver.html  (841 KB, 0.4s)
[21:35:31] ▶  explorer               running make_explorer.py...


[21:35:31] ✓  explorer               done  oil_network_explorer.html  (971 KB, 0.3s)

HTML PIPELINE SUMMARY  (total elapsed 2.6s)
  ✓ map                    ok       oil_network_map.html  (137 KB, 0.4s)
  ✓ hierarchy              ok       oil_network_hierarchy.html  (771 KB, 0.3s)
  ✓ balance                ok       oil_network_balance.html  (453 KB, 0.5s)
  ✓ map (resolver)         ok       oil_network_map_resolver.html  (136 KB, 0.3s)
  ✓ hierarchy (resolver)   ok       oil_network_hierarchy_resolver.html  (602 KB, 0.3s)
  ✓ balance (resolver)     ok       oil_network_balance_resolver.html  (841 KB, 0.4s)
  ✓ explorer               ok       oil_network_explorer.html  (971 KB, 0.3s)
